In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import OneHotEncoder, StandardScaler, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.ensemble import RandomForestClassifier

In [2]:
X, y = make_classification(n_samples=1000, n_features=15, n_informative=10, n_redundant=2,n_clusters_per_class=1, weights=[0.9], flip_y=0, random_state=42)

In [3]:
df = pd.DataFrame(X, columns=[f'fitur_{i}' for i in range(1,16)])
df["Target"]= y

In [4]:
df.head()

,fitur_1,fitur_2,fitur_3,fitur_4,fitur_5,fitur_6,fitur_7,fitur_8,fitur_9,fitur_10,fitur_11,fitur_12,fitur_13,fitur_14,fitur_15,Target
0,0.093303,-3.472520,-1.314199,3.525743,0.642138,2.247328,3.067502,1.146301,-2.173112,2.765828,-1.821258,1.459826,-1.024592,1.005559,-0.276558,0
1,-0.189574,-1.770842,-1.578851,-1.372201,-2.025230,0.518655,-0.764750,-3.958705,-0.598147,1.018789,4.194233,2.236310,-0.001984,-0.243630,0.285979,0
2,0.916269,-2.051770,3.631998,0.824844,1.674093,-0.436273,-0.460407,0.031633,-1.140149,2.069694,1.935251,0.671318,-3.175360,2.486020,-2.867291,0
3,-0.914665,-1.608657,-0.735184,-1.742743,-1.753532,0.383412,-1.057937,-2.897416,-0.830328,1.572469,5.334621,0.776033,-0.494986,-0.788215,1.255376,0
4,-0.756784,-2.362885,-3.909120,-0.474571,-4.029843,0.947114,0.581146,-3.435229,-2.142380,2.332385,3.816539,3.038337,-0.391516,0.712335,2.810524,0


In [5]:
df["fitur_12"] = np.random.choice(['A','B','C'], size=1000)
df["fitur_13"] = np.random.choice(['X','Y','Z'], size=1000)

df.head()

,fitur_1,fitur_2,fitur_3,fitur_4,fitur_5,fitur_6,fitur_7,fitur_8,fitur_9,fitur_10,fitur_11,fitur_12,fitur_13,fitur_14,fitur_15,Target
0,0.093303,-3.472520,-1.314199,3.525743,0.642138,2.247328,3.067502,1.146301,-2.173112,2.765828,-1.821258,C,Z,1.005559,-0.276558,0
1,-0.189574,-1.770842,-1.578851,-1.372201,-2.025230,0.518655,-0.764750,-3.958705,-0.598147,1.018789,4.194233,C,Z,-0.243630,0.285979,0
2,0.916269,-2.051770,3.631998,0.824844,1.674093,-0.436273,-0.460407,0.031633,-1.140149,2.069694,1.935251,A,X,2.486020,-2.867291,0
3,-0.914665,-1.608657,-0.735184,-1.742743,-1.753532,0.383412,-1.057937,-2.897416,-0.830328,1.572469,5.334621,C,Y,-0.788215,1.255376,0
4,-0.756784,-2.362885,-3.909120,-0.474571,-4.029843,0.947114,0.581146,-3.435229,-2.142380,2.332385,3.816539,A,Y,0.712335,2.810524,0


In [6]:
X = df.drop(columns=['Target'])
y = df['Target']

print(f'distribusi kelas target : {Counter(y)}')

distribusi kelas target : Counter({0: 901, 1: 99})


In [7]:
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
x_integer = X.drop(columns=['fitur_12','fitur_13'])
rf_model.fit(x_integer,y)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
important = rf_model.feature_importances_
indices = np.argsort(important)[::-1]

threshold = 0.05
important_features_indices = [i for i in range(len(important)) if important[i]  >= threshold]
'''hanya return i sebagai nilai saja bukan kolom'''


print('fitur yang dipilih')
for i in important_features_indices:
    print(f'fitur : {i}')
    
important_features = x_integer.columns[important_features_indices]

X_important = x_integer[important_features]

X_important.head()

fitur yang dipilih
fitur : 1
fitur : 2
fitur : 4
fitur : 7
fitur : 8
fitur : 9
fitur : 10
fitur : 12


,fitur_2,fitur_3,fitur_5,fitur_8,fitur_9,fitur_10,fitur_11,fitur_15
0,-3.472520,-1.314199,0.642138,1.146301,-2.173112,2.765828,-1.821258,-0.276558
1,-1.770842,-1.578851,-2.025230,-3.958705,-0.598147,1.018789,4.194233,0.285979
2,-2.051770,3.631998,1.674093,0.031633,-1.140149,2.069694,1.935251,-2.867291
3,-1.608657,-0.735184,-1.753532,-2.897416,-0.830328,1.572469,5.334621,1.255376
4,-2.362885,-3.909120,-4.029843,-3.435229,-2.142380,2.332385,3.816539,2.810524


In [9]:
X_selected = pd.concat([X_important,X['fitur_12']],axis=1)
X_selected = pd.concat([X_selected,X['fitur_13']],axis=1)

X_selected.head()


,fitur_2,fitur_3,fitur_5,fitur_8,fitur_9,fitur_10,fitur_11,fitur_15,fitur_12,fitur_13
0,-3.472520,-1.314199,0.642138,1.146301,-2.173112,2.765828,-1.821258,-0.276558,C,Z
1,-1.770842,-1.578851,-2.025230,-3.958705,-0.598147,1.018789,4.194233,0.285979,C,Z
2,-2.051770,3.631998,1.674093,0.031633,-1.140149,2.069694,1.935251,-2.867291,A,X
3,-1.608657,-0.735184,-1.753532,-2.897416,-0.830328,1.572469,5.334621,1.255376,C,Y
4,-2.362885,-3.909120,-4.029843,-3.435229,-2.142380,2.332385,3.816539,2.810524,A,Y


In [10]:
from sklearn.preprocessing import LabelEncoder
 
label_encoder = LabelEncoder()
# Melakukan Encoding untuk fitur 12
X_selected['fitur_12'] = label_encoder.fit_transform(X_selected['fitur_12'])
# print(label_encoder.inverse_transform(X_Selected[['Fitur_12']]))
# Melakukan Encoding untuk fitur 13
X_selected['fitur_13'] = label_encoder.fit_transform(X_selected['fitur_13'])
# print(label_encoder.inverse_transform(X_Selected[['Fitur_13']]))
 
print(X_selected)

      fitur_2   fitur_3   fitur_5   fitur_8   fitur_9  fitur_10  fitur_11  \
0   -3.472520 -1.314199  0.642138  1.146301 -2.173112  2.765828 -1.821258   
1   -1.770842 -1.578851 -2.025230 -3.958705 -0.598147  1.018789  4.194233   
2   -2.051770  3.631998  1.674093  0.031633 -1.140149  2.069694  1.935251   
3   -1.608657 -0.735184 -1.753532 -2.897416 -0.830328  1.572469  5.334621   
4   -2.362885 -3.909120 -4.029843 -3.435229 -2.142380  2.332385  3.816539   
..        ...       ...       ...       ...       ...       ...       ...   
995 -0.017325  4.884411  3.033376  1.278882 -1.293396  1.772169 -1.191263   
996 -1.690916  5.949207  2.238469 -1.436622 -2.153011 -0.867583  3.962758   
997 -1.263417 -1.331925  1.777577 -1.846263  1.753033  1.858452  4.264568   
998 -0.975838  0.279144 -0.443749 -1.017319 -0.643070  2.142898  3.399255   
999  1.387667 -5.554578 -2.834755 -1.932864  2.625895  0.246120  0.786726   

     fitur_15  fitur_12  fitur_13  
0   -0.276558         2         2  
1  

In [11]:
 # Memilih kolom numerik
numeric_columns = X_selected.select_dtypes(include=['float64', 'int64']).columns
numeric_columns = numeric_columns.drop(['fitur_12', 'fitur_13'])
 
# Membuat salinan data untuk menjaga data asli tetap utuh
X_cleaned = X_important.copy()

In [12]:
for col in numeric_columns:
    # Melihat outlier dengan IQR (Interquartile Range)
    Q1 = X_important[col].quantile(0.25)
    Q3 = X_important[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identifikasi outlier
    outliers = X_cleaned[(X_cleaned[col] < lower_bound) | (X_cleaned[col] > upper_bound)]
    
    # Menghapus outlier dari DataFrame
    X_important = X_important.drop(outliers.index)
    X_cleaned  = X_cleaned.drop(outliers.index)
    y          = y.drop(outliers.index)
    
    # Reset index
    X_important = X_important.reset_index(drop=True)
    X_cleaned   = X_cleaned.reset_index(drop=True)
    y           = y.reset_index(drop=True)

In [17]:
smote = SMOTE(random_state=42)

x_resampled, y_resampled = smote.fit_resample(X_cleaned,y)

x_resampled = pd.DataFrame(x_resampled)
y_resampled = pd.Series(y_resampled, name='Target')

print(Counter(y_resampled))

Counter({0: 873, 1: 873})
